In [ ]:
import os
import random
import warnings
from pathlib import Path

import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor

# ==================== 全局配置 ====================

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

# ==================== 路径配置 ====================

CURRENT_DIR = Path(__file__).parent.absolute()

DATA_DIR = CURRENT_DIR / 'data'
RESULTS_DIR = CURRENT_DIR / 'results'

MODEL_DIR = RESULTS_DIR / 'models'
PRED_DIR = RESULTS_DIR / 'predictions'
SHAP_DIR = RESULTS_DIR / 'shap_figures'
PERF_DIR = RESULTS_DIR / 'performance'

for dir_path in [RESULTS_DIR, MODEL_DIR, PRED_DIR, SHAP_DIR, PERF_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# ==================== SHAP 分析 ====================

def shap_analysis(X, y, save_dir=None):

    if save_dir is None:
        save_dir = SHAP_DIR

    print("\n开始 Random Forest SHAP 分析...")

    # 使用 Random Forest（与论文一致）
    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X, y)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # SHAP 点图
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X, show=False)

    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    plt.xlabel("SHAP Value", fontsize=13, fontweight='bold')
    plt.title("RF-SHAP Summary Plot", fontsize=14)

    plt.tight_layout()

    plt.savefig(
        save_dir / 'RF_SHAP_summary_dot.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    # SHAP 柱状图
    plt.figure(figsize=(10, 7))

    shap.summary_plot(
        shap_values,
        X,
        plot_type="bar",
        show=False
    )

    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    plt.xlabel("Mean |SHAP Value|", fontsize=13, fontweight='bold')
    plt.title("RF-SHAP Feature Importance", fontsize=14)

    plt.tight_layout()

    plt.savefig(
        save_dir / 'RF_SHAP_summary_bar.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    print(f"SHAP 图已保存至: {save_dir}")

# ==================== 保存预测结果 ====================

def save_predictions(results_df, model_name):

    save_path = PRED_DIR / f"{model_name}_LOGOCV_predictions.csv"

    results_df.to_csv(
        save_path,
        index=False,
        encoding='utf-8-sig'
    )

    print(f"预测结果已保存: {save_path}")

# ==================== LOGO-CV 模型训练 ====================

def train_and_evaluate_models(
        X,
        y,
        groups,
        models
):

    results = []

    logo = LeaveOneGroupOut()

    for name, model in models.items():

        print(f"\n{'='*60}")
        print(f"模型: {name}")
        print(f"{'='*60}")

        fold_results = []

        all_predictions = []
        all_actuals = []

        prediction_records = []

        for fold, (train_idx, test_idx) in enumerate(
                logo.split(X, y, groups)):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            fold_results.append([mse, mae, r2])

            all_predictions.extend(y_pred)
            all_actuals.extend(y_test)

            temp_df = pd.DataFrame({
                'Actual': y_test.values,
                'Predicted': y_pred,
                'Group': groups[test_idx]
            })

            prediction_records.append(temp_df)

            print(
                f"Fold {fold+1:02d} | "
                f"MSE={mse:.4f} | "
                f"MAE={mae:.4f} | "
                f"R2={r2:.4f}"
            )

        fold_results = np.array(fold_results)

        mean_mse = fold_results[:, 0].mean()
        mean_mae = fold_results[:, 1].mean()
        mean_r2 = fold_results[:, 2].mean()

        print("\nLOGO-CV 平均结果:")
        print(f"Mean MSE : {mean_mse:.4f}")
        print(f"Mean MAE : {mean_mae:.4f}")
        print(f"Mean R2  : {mean_r2:.4f}")

        results.append({
            'Model': name,
            'LOGO_MSE': mean_mse,
            'LOGO_MAE': mean_mae,
            'LOGO_R2': mean_r2
        })

        # 保存预测结果
        prediction_df = pd.concat(prediction_records)

        save_predictions(prediction_df, name)

        # 保存模型
        model.fit(X, y)

        joblib.dump(
            model,
            MODEL_DIR / f"{name}.pkl"
        )

    return pd.DataFrame(results)

# ==================== 主程序 ====================

def main():

    print("="*60)
    print("Random Forest + LOGO-CV Workflow")
    print("="*60)

    data_path = DATA_DIR / 'dataset.xlsx'

    if not data_path.exists():

        print(f"数据文件不存在: {data_path}")
        return

    data = pd.read_excel(data_path)

    print(f"\n数据形状: {data.shape}")

    # ==================== 数据定义 ====================

    group_column = data.columns[0]

    y = data.iloc[:, 1]

    X = data.iloc[:, 2:]

    groups = data[group_column]

    print(f"分组列: {group_column}")
    print(f"样本数: {len(data)}")
    print(f"特征数: {X.shape[1]}")

    # ==================== SHAP ====================

    shap_analysis(X, y)

    # ==================== 模型 ====================

    models = {

        'RandomForest': RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),

        'GradientBoost': GradientBoostingRegressor(
            n_estimators=200,
            random_state=42
        ),

        'SVR': SVR(),

        'Ridge': Ridge(alpha=1.0),

        'Adaboost': AdaBoostRegressor(
            n_estimators=200,
            random_state=42
        ),

        'DecisionTree': DecisionTreeRegressor(
            random_state=42
        ),

        'MLP': MLPRegressor(
            hidden_layer_sizes=(100,),
            max_iter=1000,
            random_state=42
        ),

        'KNN': KNeighborsRegressor(
            n_neighbors=5,
            n_jobs=-1
        ),

        'LinearRegression': LinearRegression(
            n_jobs=-1
        )
    }

    # ==================== 训练 ====================

    results_df = train_and_evaluate_models(
        X,
        y,
        groups,
        models
    )

    # ==================== 保存性能 ====================

    results_df = results_df.sort_values(
        'LOGO_R2',
        ascending=False
    )

    print("\n最终模型性能:")
    print(results_df)

    save_path = PERF_DIR / 'LOGOCV_model_performance.csv'

    results_df.to_csv(
        save_path,
        index=False,
        encoding='utf-8-sig'
    )

    print(f"\n性能结果已保存至: {save_path}")

# ==================== 运行 ====================

if __name__ == "__main__":

    main()